# Análisis gráfico de reacciones y propiedades fisicoquímicas

**Proyecto de Química – Compuestos carbonílicos**

Este notebook organiza el análisis de dos conjuntos de información:

1. **Propiedades fisicoquímicas de diez sustancias modelo:** aldehídos y cetonas.
2. **Base de 200 transformaciones químicas:** información de reacciones recopilada para el proyecto.

Se emplean tablas, estadística descriptiva y visualizaciones para explorar distribuciones y relaciones entre variables. Las gráficas describen asociaciones; por sí solas no demuestran causalidad.

**Archivo de datos requerido:** `Datos_Reacciones.xlsx`, que debe estar en la misma carpeta del notebook. En Google Colab también puede cargarse desde el computador cuando se solicite.


## 1. Importación de bibliotecas y configuración

Se utilizan Pandas y NumPy para organizar los datos; Matplotlib y Seaborn para las gráficas estáticas; y Plotly para una visualización interactiva jerárquica.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams["figure.dpi"] = 120

print("Bibliotecas importadas correctamente.")

## 2. Carga y revisión de la base de 200 transformaciones

El archivo suministrado es un libro de Excel con una hoja que contiene 200 registros y 20 variables. El código intenta leerlo desde la carpeta del notebook. Si se ejecuta en Colab y el archivo no está disponible, permite cargarlo manualmente.

**Importante:** no se generan registros aleatorios como reemplazo de los datos reales. Si el archivo no se carga, el análisis se detiene para evitar conclusiones basadas en información simulada.

In [ ]:
archivo_excel = Path("Datos_Reacciones.xlsx")

if not archivo_excel.exists():
    try:
        from google.colab import files
        print("Selecciona el archivo Datos_Reacciones.xlsx")
        archivos_subidos = files.upload()
        archivo_excel = Path(next(iter(archivos_subidos)))
    except ImportError:
        raise FileNotFoundError(
            "No se encontró Datos_Reacciones.xlsx. Coloca el archivo junto al notebook."
        )

df_cat = pd.read_excel(archivo_excel)

print("Dimensiones de la base:", df_cat.shape)
display(df_cat.head())
print("\\nColumnas disponibles:")
print(df_cat.columns.tolist())

In [ ]:
# Comprobaciones básicas de integridad
print("Número de registros:", len(df_cat))
print("Número de columnas:", len(df_cat.columns))
print("\\nValores faltantes por columna:")
display(df_cat.isna().sum().to_frame("Valores faltantes"))

if len(df_cat) != 200:
    print("Advertencia: la base no contiene exactamente 200 registros.")

## 3. Análisis de propiedades fisicoquímicas de las sustancias modelo

Esta sección conserva los valores incluidos en el código original para diez sustancias. Se comparan propiedades como peso molecular, punto de fusión, punto de ebullición, densidad, pKa, momento dipolar, solubilidad y presión de vapor.

Los valores faltantes de pKa se mantienen como `NaN`; las correlaciones se calculan con los pares de observaciones disponibles.

In [ ]:
data_num = {
    "Sustancia": [
        "Acetaldehído", "Acetona", "Benzaldehído", "Ciclohexanona", "Butanona",
        "Propanaldehído", "Formaldehído", "Acetofenona", "Benzofenona", "Cinnamaldehído"
    ],
    "Clase": [
        "Aldehído", "Cetona", "Aldehído", "Cetona", "Cetona",
        "Aldehído", "Aldehído", "Cetona", "Cetona", "Aldehído"
    ],
    "P.M.": [44.05, 58.08, 106.12, 98.14, 72.11, 58.08, 30.03, 120.15, 182.22, 132.16],
    "P.F.": [-123.5, -94.7, -26.0, -47.0, -86.3, -81.0, -92.0, 19.6, 48.5, -7.5],
    "P.Eb.": [20.2, 56.05, 178.1, 155.6, 79.6, 48.0, -19.0, 202.0, 305.4, 248.0],
    "Densidad": [0.784, 0.784, 1.044, 0.947, 0.805, 0.805, 0.815, 1.028, 1.111, 1.050],
    "pKa": [13.57, 19.20, np.nan, 18.00, 18.30, 13.10, np.nan, 18.30, np.nan, 12.50],
    "Dipolo": [2.70, 2.88, 3.02, 3.01, 2.78, 2.52, 2.33, 3.05, 2.98, 3.63],
    "Solubilidad": [1000.0, 1000.0, 6.95, 86.0, 275.0, 306.0, 400.0, 5.50, 0.14, 1.40],
    "P.Vap": [902.0, 231.0, 1.27, 4.3, 90.6, 317.0, 3890.0, 0.4, 0.001, 0.03]
}
df_num = pd.DataFrame(data_num)
display(df_num)

### 3.1. Mapa de calor de correlaciones

El mapa de calor representa el coeficiente de correlación de Pearson entre las propiedades numéricas. Los valores cercanos a +1 indican asociación lineal positiva; los cercanos a −1, asociación lineal negativa; y los próximos a 0, asociación lineal débil. La correlación no implica causalidad.

In [ ]:
cols_numericas = ["P.M.", "P.F.", "P.Eb.", "Densidad", "pKa", "Dipolo", "Solubilidad", "P.Vap"]
corr_matrix = df_num[cols_numericas].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title("Correlación entre propiedades fisicoquímicas")
plt.tight_layout()
plt.show()

### 3.2. Peso molecular frente a punto de ebullición

El diagrama de dispersión permite observar cómo se distribuyen las diez sustancias según su peso molecular y punto de ebullición. El color y el símbolo distinguen aldehídos y cetonas. La línea discontinua representa una tendencia lineal global y se utiliza únicamente como referencia exploratoria.

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(data=df_num, x="P.M.", y="P.Eb.", hue="Clase",
                     style="Clase", s=120)
sns.regplot(data=df_num, x="P.M.", y="P.Eb.", scatter=False,
            color="gray", line_kws={"linestyle": "--"})
for _, fila in df_num.iterrows():
    ax.annotate(fila["Sustancia"], (fila["P.M."], fila["P.Eb."]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
plt.xlabel("Peso molecular (g/mol)")
plt.ylabel("Punto de ebullición (°C)")
plt.title("Peso molecular frente a punto de ebullición")
plt.tight_layout()
plt.show()

### 3.3. Distribución de densidad y momento dipolar por clase

El diagrama de cajas resume la distribución de densidad y permite observar la dispersión de los valores individuales. El gráfico de violín representa la distribución del momento dipolar. Debido al tamaño reducido de la muestra (cinco sustancias por clase), estas figuras se interpretan de manera descriptiva y no como evidencia concluyente de diferencias entre familias.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df_num, x="Clase", y="Densidad", ax=axes[0], width=0.45)
sns.stripplot(data=df_num, x="Clase", y="Densidad", ax=axes[0],
              color="black", size=6)
axes[0].set_title("Distribución de densidad")
axes[0].set_ylabel("Densidad (g/cm³)")

sns.violinplot(data=df_num, x="Clase", y="Dipolo", ax=axes[1],
               inner="point", cut=0)
sns.stripplot(data=df_num, x="Clase", y="Dipolo", ax=axes[1],
              color="black", size=4)
axes[1].set_title("Distribución del momento dipolar")
axes[1].set_ylabel("Momento dipolar (D)")
plt.tight_layout()
plt.show()

### 3.4. Distribución del punto de fusión y solubilidad

El histograma permite examinar la distribución de los puntos de fusión. El gráfico de solubilidad compara los valores de cada sustancia; se ordenan por peso molecular para facilitar la lectura. Como la solubilidad presenta una amplitud considerable, conviene evitar interpretar las distancias visuales como diferencias uniformes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df_num["P.F."], bins=6, kde=True, ax=axes[0])
axes[0].set_title("Distribución del punto de fusión")
axes[0].set_xlabel("Punto de fusión (°C)")

df_sorted_sol = df_num.sort_values("P.M.")
axes[1].bar(df_sorted_sol["Sustancia"], df_sorted_sol["Solubilidad"])
axes[1].tick_params(axis="x", rotation=70)
axes[1].set_title("Solubilidad acuosa por sustancia")
axes[1].set_ylabel("Solubilidad (g/L)")
plt.tight_layout()
plt.show()

## 4. Análisis categórico de las 200 transformaciones

En esta sección se emplean las categorías registradas en el archivo Excel. Se verifica que las columnas requeridas existan antes de construir las visualizaciones. Las variables principales son `Tipo_Reaccion`, `Medio` y `Temperatura_C`.

La base también distingue `Tipo_Transformacion` (por ejemplo, reacción u obtención), por lo que puede utilizarse para contextualizar el conjunto de registros.

In [ ]:
columnas_requeridas = ["Tipo_Reaccion", "Medio", "Temperatura_C"]
faltantes = [c for c in columnas_requeridas if c not in df_cat.columns]
if faltantes:
    raise ValueError(f"Faltan columnas necesarias en el Excel: {faltantes}")

# Normalización mínima de tipos para los gráficos
for col in ["Tipo_Reaccion", "Medio", "Tipo_Transformacion"]:
    if col in df_cat.columns:
        df_cat[col] = df_cat[col].astype("string").str.strip()

display(df_cat["Tipo_Transformacion"].value_counts(dropna=False).to_frame("Frecuencia"))

### 4.1. Frecuencia por tipo de reacción

El gráfico de barras cuenta cuántos registros corresponden a cada categoría de `Tipo_Reaccion`. Permite identificar cuáles son las transformaciones más frecuentes dentro de la base recopilada.

In [ ]:
conteo_reacciones = df_cat["Tipo_Reaccion"].value_counts().sort_values(ascending=True)
plt.figure(figsize=(10, 6))
conteo_reacciones.plot(kind="barh", edgecolor="black")
plt.xlabel("Número de registros")
plt.ylabel("Tipo de reacción")
plt.title("Frecuencia de transformaciones por tipo de reacción")
plt.tight_layout()
plt.show()

### 4.2. Distribución del medio de reacción

El gráfico circular muestra la proporción de registros clasificados según el medio consignado en el Excel. Las categorías se presentan tal como aparecen en la base; por ello, no se agrupan automáticamente categorías como ácido, básico, neutro o apolar.

In [ ]:
conteo_medio = df_cat["Medio"].value_counts(dropna=False)
plt.figure(figsize=(7, 7))
plt.pie(conteo_medio.values, labels=conteo_medio.index.astype(str),
        autopct="%1.1f%%", startangle=140)
plt.title("Distribución de registros por medio de reacción")
plt.tight_layout()
plt.show()

### 4.3. Distribución de condiciones térmicas

El Excel registra la temperatura como variable numérica. Para facilitar su lectura se crean intervalos descriptivos: enfriamiento (≤ 0 °C), temperatura ambiente (1–25 °C), temperatura intermedia (26–60 °C) y calentamiento (> 60 °C). Estos intervalos son una clasificación operativa para la visualización y no categorías originales de SciFinder-n.

In [ ]:
temperatura = pd.to_numeric(df_cat["Temperatura_C"], errors="coerce")
bins = [-np.inf, 0, 25, 60, np.inf]
labels = ["Enfriamiento (≤ 0 °C)", "Ambiente (1–25 °C)",
          "Intermedia (26–60 °C)", "Calentamiento (> 60 °C)"]
df_cat["Categoria_Termica"] = pd.cut(temperatura, bins=bins, labels=labels)

conteo_temp = df_cat["Categoria_Termica"].value_counts(sort=False).dropna()
plt.figure(figsize=(9, 5))
plt.pie(conteo_temp.values, labels=conteo_temp.index.astype(str),
        autopct="%1.1f%%", startangle=90,
        wedgeprops={"width": 0.42})
plt.title("Distribución por intervalos de temperatura")
plt.tight_layout()
plt.show()

display(conteo_temp.to_frame("Frecuencia"))

### 4.4. Diagrama Sunburst: tipo de reacción, medio y categoría térmica

El diagrama Sunburst es interactivo. Cada nivel representa una variable categórica y permite explorar cómo se distribuyen los registros dentro de las jerarquías. Se excluyen únicamente los registros que no tengan información en alguna de las tres variables utilizadas.

In [ ]:
df_sun = df_cat.dropna(subset=["Tipo_Reaccion", "Medio", "Categoria_Termica"]).copy()

if not df_sun.empty:
    fig_sun = px.sunburst(
        df_sun,
        path=["Tipo_Reaccion", "Medio", "Categoria_Termica"],
        title="Jerarquía: tipo de reacción → medio → categoría térmica"
    )
    fig_sun.show()
else:
    print("No hay registros completos suficientes para construir el Sunburst.")

## 5. Resumen descriptivo y control de calidad

Esta sección presenta frecuencias y estadísticos de temperatura. Estos resultados ayudan a comprobar la base y sirven como apoyo para redactar la interpretación final.

In [ ]:
print("Frecuencia por tipo de reacción:")
display(df_cat["Tipo_Reaccion"].value_counts(dropna=False).to_frame("Frecuencia"))

print("Frecuencia por medio:")
display(df_cat["Medio"].value_counts(dropna=False).to_frame("Frecuencia"))

print("Resumen de temperatura (°C):")
display(pd.to_numeric(df_cat["Temperatura_C"], errors="coerce").describe().to_frame("Temperatura_C"))



**Conclusión:** el análisis gráfico permite resumir las propiedades fisicoquímicas de las sustancias modelo y explorar la distribución de las transformaciones químicas registradas. La interpretación final debe relacionar cada patrón observado con la estructura molecular, las condiciones de reacción y las limitaciones de los datos, evitando generalizaciones que la muestra no permita sostener.
